In [0]:
import csv

# Column headers
headers = [
    "User ID", "Card ID", "Account Open Date", "Amount", "Card Type", 
    "Card Expiry Date", "Total transactions on the card", 
    "Card Limit", "Transaction Date", "Transaction Type", 
    "Merchant City", "Merchant State"
]

# Sample data
data = [
    [1, 12345, "2023-01-01", 100.00, "Visa", "2025-01-01", 5, 1000.00, "2023-01-01", "Purchase", "New York", "NY"],
    [3, 14897, "2023-02-01", 200.00, "Platnium", "2025-02-01", 7, 2000.00, "2023-02-01", "Purchase", "Rochester", "NY"],
    [5, 24688, "2023-03-01", 400.00, "Master", "2025-03-01", 7, 4000.00, "2023-03-01", "Swipe_Transaction", "Rochester", "MA"],
    [7, 357911, "2023-04-01", 600.00, "Master", "2025-04-03", 7, 6000.00, "2023-04-02", "Swipe_Transaction", "Boston", "MA"],
    [9, 908765, "2023-06-01", 800.00, "Visa", "2025-06-10", 7, 8000.00, "2023-05-10", "Swipe_Transaction", "Dallas", "TX"],
    [6, 467859, "2023-12-01", 1000.00, "Master", "2025-10-12", 7, 10000.00, "2023-10-10", "Swipe_Transaction", "Carollton", "TX"]
    # Add more rows as needed
]

# Writing to CSV file
with open('transactions.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(data)


In [0]:
import csv

csv_file_path = "transactions.csv"

with open(csv_file_path, mode='r') as file:
    reader = csv.reader(file)
    
    for row in reader:
        print(row)


['User ID', 'Card ID', 'Account Open Date', 'Amount', 'Card Type', 'Card Expiry Date', 'Total transactions on the card', 'Card Limit', 'Transaction Date', 'Transaction Type', 'Merchant City', 'Merchant State']
['1', '12345', '2023-01-01', '100.0', 'Visa', '2025-01-01', '5', '1000.0', '2023-01-01', 'Purchase', 'New York', 'NY']
['3', '14897', '2023-02-01', '200.0', 'Platnium', '2025-02-01', '7', '2000.0', '2023-02-01', 'Purchase', 'Rochester', 'NY']
['5', '24688', '2023-03-01', '400.0', 'Master', '2025-03-01', '7', '4000.0', '2023-03-01', 'Swipe_Transaction', 'Rochester', 'MA']
['7', '357911', '2023-04-01', '600.0', 'Master', '2025-04-03', '7', '6000.0', '2023-04-02', 'Swipe_Transaction', 'Boston', 'MA']
['9', '908765', '2023-06-01', '800.0', 'Visa', '2025-06-10', '7', '8000.0', '2023-05-10', 'Swipe_Transaction', 'Dallas', 'TX']
['6', '467859', '2023-12-01', '1000.0', 'Master', '2025-10-12', '7', '10000.0', '2023-10-10', 'Swipe_Transaction', 'Carollton', 'TX']


In [0]:
from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder.appName("example").getOrCreate()

# Sample data for dim_phones_apple
phones_data = [("iPhone X", "Apple", 999),
               ("iPhone 11", "Apple", 1099),
               ("iPhone 12", "Apple", 1199)]

# Sample data for dim_phoneOrders_apple
phone_orders_data = [(1, "John", "iPhone X", "2023-01-01", 1),
                     (2, "Jane", "iPhone 11", "2023-01-02", 2),
                     (3, "Bob", "iPhone 12", "2023-01-03", 1)]

# Define schema for dim_phones_apple
phones_schema = ["model", "brand", "price"]
phone_orders_schema = ["order_id", "customer_name", "phone_model", "order_date", "quantity"]

# Create dataframes for dim_phones_apple and dim_phoneOrders_apple
dim_phones_sold_apple_df = spark.createDataFrame(phones_data, schema=phones_schema)
dim_phoneOrders_placed_apple_df = spark.createDataFrame(phone_orders_data, schema=phone_orders_schema)

# Register dataframes as temporary tables
dim_phones_sold_apple_df.createOrReplaceTempView("dim_phones_apple")
dim_phoneOrders_placed_apple_df.createOrReplaceTempView("dim_phoneOrders_apple")

# Save dataframes to CSV files
dim_phones_sold_apple_df.write.csv("dim_phones_sold_apple.csv", header=True)
dim_phoneOrders_placed_apple_df.write.csv("dim_phoneOrders_placed_apple.csv", header=True)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-2912189274889867>:29
     26 dim_phoneOrders_placed_apple_df.createOrReplaceTempView("dim_phoneOrders_apple")
     28 # Save dataframes to CSV files
---> 29 dim_phones_sold_apple_df.write.csv("dim_phones_sold_apple.csv", header=True)
     30 dim_phoneOrders_placed_apple_df.write.csv("dim_phoneOrders_placed_apple.csv", header=True)

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1798, in DataFrameWriter.csv(self, path, mode, compression, sep, quote,

In [0]:
%sql
SELECT
    doa.order_id,
    doa.customer_id,
    doa.phone_id,
    doa.order_time,
    doa.order_status,
    dpa.phone_id,
    TRIM(SPLIT(dpa.phone_names, '[-:*~]')[0]) AS brand,
    TRIM(SPLIT(dpa.phone_names, '[-:*~]')[1]) AS model_name,
    dpa.price_usd
FROM
    dim_phoneOrders_apple doa
JOIN
    dim_phones_apple dpa
ON
    doa.phone_id = dpa.phone_id;


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-1619340249697140>:7
      5     display(df)
      6     return df
----> 7   _sqldf = ____databricks_percent_sql()
      8 finally:
      9   del ____databricks_percent_sql

File <command-1619340249697140>:4, in ____databricks_percent_sql()
      2 def ____databricks_percent_sql():
      3   import base64
----> 4   df = spark.sql(base64.standard_b64decode("U0VMRUNUCiAgICBkb2Eub3JkZXJfaWQsCiAgICBkb2EuY3VzdG9tZXJfaWQsCiAgICBkb2EucGhvbmVfaWQsCiAgICBkb2Eub3JkZXJfdGltZSwKICAgIGRvYS5vcmRlcl9zdGF0dXMsCiAgICBkcGEucGhvbmVfaWQsCiAgICBUUklNKFNQTElUKGRwYS5waG9uZV9uYW1lcywgJ1stOip+XScpWzBdKSBBUyBicmFuZCwKICAgIFRSSU0oU1BMSVQoZHBhLnBob25lX25hbWVzLCAnWy06Kn5dJylbMV0pIEFTIG1vZGVsX25hbWUsCiAgICBkcGEucHJpY2VfdXNkCkZST00KICAgIGRpbV9waG9uZU9yZGVyc19hcHBsZSBkb2EKSk9JTgogICAgZGltX3Bob25lc19hcHBsZSBkcGEKT04KICAgIGRvYS5waG9uZV9pZCA9IGRwYS5waG9uZV9p